# Multiple Regression

In [1]:
import polars as pl
import numpy as np
import hvplot.polars
import holoviews as hv
from datascience import make_array, minimize
path_data = '../../../assets/data/'

np.set_printoptions(suppress=True)

In [2]:
def standard_units(any_numbers):
    "Convert any column (a Polars Series) of numbers to standard units."
    return (any_numbers - any_numbers.mean())/any_numbers.std(ddof=0)

def correlation(t, x, y):
    return (standard_units(t.get_column(x))*standard_units(t.get_column(y))).mean()

Now that we have explored ways to use multiple attributes to predict a categorical variable, let us return to predicting a quantitative variable. Predicting a numerical quantity is called regression, and a commonly used method to use multiple attributes for regression is called *multiple linear regression*.

## Home Prices

The following dataset of house prices and attributes was collected over several years for the city of Ames, Iowa. A [description of the dataset appears online](http://ww2.amstat.org/publications/jse/v19n3/decock.pdf). We will focus only a subset of the columns. We will try to predict the sale price column from the other columns.

In [3]:
all_sales = pl.read_csv(path_data + 'house.csv')
sales = all_sales.filter(pl.col('Bldg Type') == '1Fam').filter(pl.col('Sale Condition') == 'Normal').select(
    'SalePrice', '1st Flr SF', '2nd Flr SF', 
    'Total Bsmt SF', 'Garage Area', 
    'Wood Deck SF', 'Open Porch SF', 'Lot Area', 
    'Year Built', 'Yr Sold')
sales.sort('SalePrice')

SalePrice,1st Flr SF,2nd Flr SF,Total Bsmt SF,Garage Area,Wood Deck SF,Open Porch SF,Lot Area,Year Built,Yr Sold
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
35000,498,0,498,216,0,0,8088,1922,2006
39300,334,0,0,0,0,0,5000,1946,2007
40000,649,668,649,250,0,54,8500,1920,2008
45000,612,0,0,308,0,0,5925,1940,2009
52000,729,0,270,0,0,0,4130,1935,2008
…,…,…,…,…,…,…,…,…,…
584500,1933,1567,1733,959,870,86,17242,1993,2006
610000,2674,0,2630,762,360,50,13693,2007,2009
615000,2470,0,2535,789,154,65,12720,2003,2008


A histogram of sale prices shows a large amount of variability and a distribution that is clearly not normal. A long tail to the right contains a few houses that had very high prices. The short left tail does not contain any houses that sold for less than $35,000.

In [4]:
sales.hvplot.hist('SalePrice', bins=32, normed=True, xlabel='SalePrice ($)')

Histogram with 'SalePrice ($)' on the x-axis and 'Percent per $' on the y-axis. The histogram has bars of increasing size up to about x=120000 and then the bars have decreasing height visible through about 500000 with the graph extending out to 700000.

### Correlation

No single attribute is sufficient to predict the sale price. For example, the area of first floor, measured in square feet, correlates with sale price but only explains some of its variability.

In [5]:
sales.hvplot.scatter(x='1st Flr SF', y='SalePrice')

Scatterplot with '1st Flr SF' on the x-axis and 'Sale Price' on the y-axis. The data have a positive association, fanning out as the x values increase.

In [6]:
correlation(sales, 'SalePrice', '1st Flr SF')

0.6424662541030224

In fact, none of the individual attributes have a correlation with sale price that is above 0.7 (except for the sale price itself).

In [7]:
for label in sales.columns:
    print('Correlation of', label, 'and SalePrice:\t', correlation(sales, label, 'SalePrice'))

Correlation of SalePrice and SalePrice:	 1.0000000000000002
Correlation of 1st Flr SF and SalePrice:	 0.6424662541030224
Correlation of 2nd Flr SF and SalePrice:	 0.3575218942800824
Correlation of Total Bsmt SF and SalePrice:	 0.6529786267571691
Correlation of Garage Area and SalePrice:	 0.6385944852520443
Correlation of Wood Deck SF and SalePrice:	 0.3526986661950492
Correlation of Open Porch SF and SalePrice:	 0.3369094170263733
Correlation of Lot Area and SalePrice:	 0.2908234551157694
Correlation of Year Built and SalePrice:	 0.5651647537135915
Correlation of Yr Sold and SalePrice:	 0.02594857908072063


However, combining attributes can provide higher correlation. In particular, if we sum the first floor and second floor areas, the result has a higher correlation than any single attribute alone.

In [8]:
both_floors = sales.to_series(1) + sales.to_series(2)
correlation(sales.with_columns(both_floors.alias('Both Floors')), 'SalePrice', 'Both Floors')

0.7821920556134876

This high correlation indicates that we should try to use more than one attribute to predict the sale price. In a dataset with multiple observed attributes and a single numerical value to be predicted (the sale price in this case), multiple linear regression can be an effective technique.

## Multiple Linear Regression 

In multiple linear regression, a numerical output is predicted from numerical input attributes by multiplying each attribute value by a different slope, then summing the results. In this example, the slope for the `1st Flr SF` would represent the dollars per square foot of area on the first floor of the house that should be used in our prediction. 

Before we begin prediction, we split our data randomly into a training and test set of equal size.

In [9]:
# POLARS_CHANGE: Table.split shuffles the rows and splits them in two. Shuffle with sample, then take head and tail.
shuffled_sales = sales.sample(n=sales.height, with_replacement=False, shuffle=True)
train = shuffled_sales.head(1001)
test = shuffled_sales.tail(sales.height - 1001)
print(train.height, 'training and', test.height, 'test instances.')

1001 training and 1001 test instances.


The slopes in multiple regression is an array that has one slope value for each attribute in an example. Predicting the sale price involves multiplying each attribute by the slope and summing the result.

In [10]:
def predict(slopes, row):
    return sum(slopes * np.array(row))

example_row = test.drop('SalePrice').row(0)
print('Predicting sale price for:', example_row)
example_slopes = np.random.normal(10, 1, len(example_row))
print('Using slopes:', example_slopes)
print('Result:', predict(example_slopes, example_row))

Predicting sale price for: (1640, 0, 0, 394, 0, 0, 21370, 1950, 2006)
Using slopes: [  9.66907864  10.41089808   7.65174362  11.88933605  10.82544589
  10.39099739  10.53103519   9.08003219   7.53191751]
Result: 278404.998677


The result is an estimated sale price, which can be compared to the actual sale price to assess whether the slopes provide accurate predictions. Since the `example_slopes` above were chosen at random, we should not expect them to provide accurate predictions at all.

In [11]:
print('Actual sale price:', test.get_column('SalePrice').item(0))
print('Predicted sale price using random slopes:', predict(example_slopes, example_row))

Actual sale price: 131000
Predicted sale price using random slopes: 278404.998677


### Least Squares Regression

The next step in performing multiple regression is to define the least squares objective. We perform the prediction for each row in the training set, and then compute the root mean squared error (RMSE) of the predictions from the actual prices.

In [12]:
train_prices = train.to_series(0)
train_attributes = train.drop('SalePrice')

def rmse(slopes, attributes, prices):
    errors = []
    for i in np.arange(len(prices)):
        predicted = predict(slopes, attributes.row(i))
        actual = prices.item(i)
        errors.append((predicted - actual) ** 2)
    return np.mean(errors) ** 0.5

def rmse_train(slopes):
    return rmse(slopes, train_attributes, train_prices)

print('RMSE of all training examples using random slopes:', rmse_train(example_slopes))

RMSE of all training examples using random slopes: 69690.253371


Finally, we use the `minimize` function to find the slopes with the lowest RMSE. Since the function we want to minimize, `rmse_train`, takes an array instead of a number, we must pass the `array=True` argument to `minimize`. When this argument is used, `minimize` also requires an initial guess of the slopes so that it knows the dimension of the input array. Finally, to speed up optimization, we indicate that `rmse_train` is a smooth function using the `smooth=True` attribute. Computation of the best slopes may take several minutes.

In [13]:
best_slopes = minimize(rmse_train, start=example_slopes, smooth=True, array=True)
print('The best slopes for the training set:')
# POLARS_CHANGE: reshape(1, -1) makes best_slopes one row of a DataFrame; tbl_cols=-1 shows every column.
pl.from_numpy(best_slopes.reshape(1, -1), schema=train_attributes.columns).show(tbl_cols=-1)
print('RMSE of all training examples using the best slopes:', rmse_train(best_slopes))

The best slopes for the training set:

1st Flr SF,2nd Flr SF,Total Bsmt SF,Garage Area,Wood Deck SF,Open Porch SF,Lot Area,Year Built,Yr Sold
f64,f64,f64,f64,f64,f64,f64,f64,f64
79.489728,77.668228,45.269151,39.313836,40.313353,9.736304,0.48802,535.211283,-531.228645


RMSE of all training examples using the best slopes: 31897.7483008


### Interpreting Multiple Regression 

Let's interpret these results. The best slopes give us a method for estimating the price of a house from its attributes. A square foot of area on the first floor is worth about \$75 (the first slope), while one on the second floor is worth about \$70 (the second slope). The final negative value describes the market: prices in later years were lower on average.

The RMSE of around \$30,000 means that our best linear prediction of the sale price based on all of the attributes is off by around \$30,000 on the training set, on average.  We find a similar error when predicting prices on the test set, which indicates that our prediction method will generalize to other samples from the same population.

In [14]:
test_prices = test.to_series(0)
test_attributes = test.drop('SalePrice')

def rmse_test(slopes):
    return rmse(slopes, test_attributes, test_prices)

rmse_linear = rmse_test(best_slopes)
print('Test set RMSE for multiple linear regression:', rmse_linear)

Test set RMSE for multiple linear regression: 30344.1211834


If the predictions were perfect, then a scatter plot of the predicted and actual values would be a straight line with slope 1. We see that most dots fall near that line, but there is some error in the predictions.

In [15]:
def fit(row):
    return sum(best_slopes * np.array(row))

# POLARS_CHANGE: iter_rows passes each test row as a tuple; collect the fitted values in an array.
fitted = make_array()
for row in test.drop('SalePrice').iter_rows():
    fitted = np.append(fitted, fit(row))

(
    test.with_columns(pl.Series('Fitted', fitted)).hvplot.scatter(x='Fitted', y='SalePrice')
    * hv.Curve(([0, 5e5], [0, 5e5])).opts(color='#006a94', line_width=2)
)

Scatterplot with 'Fitted' on the x-axis and 'Sale Price' on the y-axis. The data appears to have a fairly strong positive correlation and a light blue line is drawn over most of the data. As the x values get higher however, the data points begin to be further away from the line.

A residual plot for multiple regression typically compares the errors (residuals) to the actual values of the predicted variable. We see in the residual plot below that we have systematically underestimated the value of expensive houses, shown by the many positive residual values on the right side of the graph.

In [16]:
(
    test.with_columns((test_prices - fitted).alias('Residual')).hvplot.scatter(x='SalePrice', y='Residual')
    * hv.Curve(([0, 7e5], [0, 0])).opts(color='#006a94', line_width=2)
)

Scatterplot with 'Sale Price' on the x-axis and 'Residual' on the y-axis. There is a horizontal light blue line drawn at y=0. The data points below x=300000 generally are spread evenly above and below the light blue line. However, above x=300000, there are more data points above the line than below it. By x=500000, all data points are well above the line.

As with simple linear regression, interpreting the result of a predictor is at least as important as making predictions. There are many lessons about interpreting multiple regression that are not included in this textbook. A natural next step after completing this text would be to study linear modeling and regression in further depth.

## Nearest Neighbors for Regression

Another approach to predicting the sale price of a house is to use the price of similar houses. This *nearest neighbor* approach is very similar to our classifier. To speed up computation, we will only use the attributes that had the highest correlation with the sale price in our original analysis.

In [17]:
train_nn = train.select(pl.nth(0, 1, 2, 3, 4, 8))
test_nn = test.select(pl.nth(0, 1, 2, 3, 4, 8))
train_nn.show(3)

SalePrice,1st Flr SF,2nd Flr SF,Total Bsmt SF,Garage Area,Year Built
i64,i64,i64,i64,i64,i64
222500,884,1151,884,434,2001
157500,720,720,720,480,1964
174000,744,700,384,400,2006


The computation of closest neighbors is identical to a nearest-neighbor classifier. In this case, we will exclude the `'SalePrice'` rather than the `'Class'` column from the distance computation. The five nearest neighbors of the first test row are shown below.

In [18]:
def distance(pt1, pt2):
    """The distance between two points, represented as arrays."""
    return np.sqrt(sum((pt1 - pt2) ** 2))

def row_distance(row1, row2):
    """The distance between two rows of a table."""
    return distance(np.array(row1), np.array(row2))

def distances(training, example, output):
    """Compute the distance from example for each row in training."""
    dists = make_array()
    attributes = training.drop(output)
    for row in attributes.iter_rows():
        dists = np.append(dists, row_distance(row, example))
    return training.with_columns(pl.Series('Distance', dists))

def closest(training, example, k, output):
    """Return a table of the k closest neighbors to example."""
    return distances(training, example, output).sort('Distance').head(k)

example_nn_row = test_nn.drop('SalePrice').row(0)
closest(train_nn, example_nn_row, 5, 'SalePrice')

SalePrice,1st Flr SF,2nd Flr SF,Total Bsmt SF,Garage Area,Year Built,Distance
i64,i64,i64,i64,i64,i64,f64
139600,1733,0,0,452,1955,109.71782
81500,1600,0,0,270,1951,130.295817
115000,1771,0,0,336,1960,143.614066
135000,1488,0,0,569,1952,231.803796
110000,1382,0,0,384,1949,258.195662


One simple method for predicting the price is to average the prices of the nearest neighbors.

In [19]:
def predict_nn(example):
    """Return the majority class among the k nearest neighbors."""
    return closest(train_nn, example, 5, 'SalePrice').get_column('SalePrice').mean()

predict_nn(example_nn_row)

116220.0

Finally, we can inspect whether our prediction is close to the true sale price for our one test example. Looks reasonable!

In [20]:
print('Actual sale price:', test_nn.get_column('SalePrice').item(0))
print('Predicted sale price using nearest neighbors:', predict_nn(example_nn_row))

Actual sale price: 131000
Predicted sale price using nearest neighbors: 116220.0


### Evaluation

To evaluate the performance of this approach for the whole test set, we apply `predict_nn` to each test example, then compute the root mean squared error of the predictions. Computation of the predictions may take several minutes.

In [21]:
nn_test_predictions = make_array()
for row in test_nn.drop('SalePrice').iter_rows():
    nn_test_predictions = np.append(nn_test_predictions, predict_nn(row))
rmse_nn = ((test_prices - nn_test_predictions) ** 2).mean() ** 0.5

print('Test set RMSE for multiple linear regression: ', rmse_linear)
print('Test set RMSE for nearest neighbor regression:', rmse_nn)

Test set RMSE for multiple linear regression:  30344.1211834
Test set RMSE for nearest neighbor regression: 31143.785035114895


For these data, the errors of the two techniques are quite similar! For different data sets, one technique might outperform another. By computing the RMSE of both techniques on the same data, we can compare methods fairly. One note of caution: the difference in performance might not be due to the technique at all; it might be due to the random variation due to sampling the training and test sets in the first place.

Finally, we can draw a residual plot for these predictions. We still underestimate the prices of the most expensive houses, but the bias does not appear to be as systematic. However, fewer residuals are very close to zero, indicating that fewer prices were predicted with very high accuracy. 

In [22]:
(
    test.with_columns((test_prices - nn_test_predictions).alias('Residual')).hvplot.scatter(x='SalePrice', y='Residual')
    * hv.Curve(([0, 7e5], [0, 0])).opts(color='#006a94', line_width=2)
)

Scatterplot with 'Sale Price' on the x-axis and 'Residual' on the y-axis. The data points have a positive association, but most exist within a blob on the left hand side of the graph. A horizontal light blue line is drawn at y=0. The right hand side of the graph has more data points above the line than below it.